In [1]:
import pandas as pd
import spacy
import re 
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz
import numpy as np
import unicodedata 
from spacy.matcher import PhraseMatcher

In [2]:
dir = '/Users/emudr/PPMI_LEDD/data/ADNI/'
raw_file = 'raw/RECCMEDS_25Mar2026.csv'
df = pd.read_csv(dir + raw_file, low_memory = False)

cols = ["CMMED", "CMDOSE", "CMUNITS", "CMUNITO", "CMFREQNCO"]
df[cols] = df[cols].replace(-4, np.nan)
df[cols] = df[cols].replace('-4', np.nan)


unit_map = {
    1: "drop",
    2: "gram",
    3: "international units",
    4: "microgram",
    5: "milligram",
    6: "milliliter",
    7: "percent",
    8: "puff",
    9: "spray",
    10: "tablespoon",
    11: "tabs",
    12: "teaspoon"
}

df["CMUNITS"] = df["CMUNITS"].map(unit_map)

med_list = 'common_meds.csv'
med_df = pd.read_csv(dir + med_list)
med_list = med_df['common_meds'].to_list()



In [3]:
def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s/]', ' ', text)   # keep slash for combos
    text = re.sub(r'\s+', ' ', text).strip()
    return text



def match_med(text, med_list, threshold=70):
    text = normalize(text)
    
    if not text:
        return None
    
    match, score, _ = process.extractOne(
        text,
        med_list,
        scorer=fuzz.token_set_ratio  # best for messy text
    )
    
    if score >= threshold:
        return match
    return None


def combine_row(row):
    return " ".join(
        str(x) for x in row
        if pd.notna(x) and str(x).strip() != ""
    )

med_list = [normalize(m) for m in med_list]
df["med_match"] = df["CMMED"].apply(lambda x: match_med(x, med_list))
filtered_df = df[df["med_match"].notna()]

cols = ["CMMED", "CMDOSE", "CMUNITS", "CMUNITO", "CMFREQNCO"]
filtered_df[cols] = df[cols].replace(-4, np.nan)
filtered_df["CMMED_simulated"] = df[cols].apply(combine_row, axis=1)
filtered_df.to_csv(dir + 'simulated/RECCMEDS_25Mar2026_simulated.csv')

C:\Users\emudr\AppData\Local\Temp\ipykernel_20008\1302426745.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df[cols] = df[cols].replace(-4, np.nan)
C:\Users\emudr\AppData\Local\Temp\ipykernel_20008\1302426745.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["CMMED_simulated"] = df[cols].apply(combine_row, axis=1)
